In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    # Resolve imports from either the repository root or notebooks directory.
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from src.analysis.presentation import format_analysis_table
from src.analysis.orders import build_order_analysis_view
from src.analysis.summary import summarize_delivery_by_slot
from src.analysis.statistics import test_slot_association

from src.analysis.plots import (
    build_temporal_profiles, plot_slot_counts, plot_rate_profile,
    plot_hour_weekday_heatmap, plot_delay_distribution,
)
from src.analysis.route_slots import build_state_pair_orders, summarize_state_pair_slots
from src.analysis.route_stages import compare_route_delivery_stages, summarize_afternoon_route_evidence
from src.data_loader import load_data
from src.paths import PROCESSED_DIR

from src.analysis.investigation import (
    summarize_time_periods, compare_cleaning_cohorts, investigate_delivery_stages,
    classify_order_routes, summarize_routes, build_focus_cohort, summarize_destination_states,
    summarize_state_pairs, summarize_focus_time, summarize_focus_comparisons,
)
from src.analysis.investigation_plots import (
    plot_monthly_slot_rates, plot_matched_years, plot_stage_comparison, plot_promised_windows,
    plot_route_slot_rates, plot_route_transit_time, plot_destination_states, plot_state_pairs,
    plot_focus_months, plot_focus_approval_time,
    plot_state_pair_slots, plot_state_pair_months,
    plot_route_period_medians,
    plot_route_period_slot_rates,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed input: {PROCESSED_DIR}")


In [ ]:
import pandas as pd
import numpy as np

from src.paths import PROCESSED_DIR, INTERIM_DIR

pd.set_option("display.max_columns", None)

In [ ]:
# Find the analysis dataset
candidates = [
    PROCESSED_DIR / "order_analysis.parquet",
    INTERIM_DIR / "order_analysis.parquet"
]

data_path = next(
    (path for path in candidates if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError("order_analysis.parquet not found")

df = pd.read_parquet(data_path)

print("Shape:", df.shape)
display(df.head())

In [ ]:
df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"]
)

df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"]
)

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

# Recalculate target based on calendar dates
recalculated_late = (
    df["order_delivered_customer_date"].dt.normalize()
    > df["order_estimated_delivery_date"].dt.normalize()
).astype(int)

# Compare with existing label
mismatches = (
    recalculated_late != df["is_late"].astype(int)
).sum()

print("Label mismatches:", mismatches)
print("\nTarget distribution:")
print(df["is_late"].value_counts())
print("\nTarget percentage:")
print(df["is_late"].value_counts(normalize=True) * 100)

assert mismatches == 0
assert df["order_id"].is_unique

In [ ]:
# Numeric features already available
numeric_features = [
    "order_value",
    "freight_value",
    "item_count",
    "n_sellers",
    "total_weight_g"
]

# Categorical features
categorical_features = [
    "customer_state",
    "seller_state",
    "product_category"
]

# Initialize modeling dataset
model_df = df[
    ["order_id", "order_purchase_timestamp"]
    + numeric_features
    + categorical_features
].copy()

# Estimated delivery lead time (days)
model_df["estimated_lead_days"] = (
    df["order_estimated_delivery_date"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Purchase-time features
model_df["purchase_hour"] = (
    df["order_purchase_timestamp"].dt.hour
)

model_df["purchase_weekday"] = (
    df["order_purchase_timestamp"].dt.day_name()
)

model_df["purchase_month"] = (
    df["order_purchase_timestamp"].dt.month
)

# Geographic feature
assert df[["customer_state", "seller_state"]].notna().all().all()

model_df["same_state"] = (
    df["customer_state"] == df["seller_state"]
).astype(int)

# Target
model_df["is_late"] = df["is_late"].astype(int)

print("Shape:", model_df.shape)
display(model_df.head())

In [ ]:
print("Missing values:")
display(
    model_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("Duplicate orders:",
      model_df["order_id"].duplicated().sum())

print("Target distribution:")
display(
    model_df["is_late"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Sort chronologically
model_df = model_df.sort_values(
    ["order_purchase_timestamp", "order_id"]
).reset_index(drop=True)

n = len(model_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Assign split
model_df["split"] = "test"

model_df.loc[:train_end - 1, "split"] = "train"
model_df.loc[train_end:val_end - 1, "split"] = "validation"

# Summarize distribution
split_summary = model_df.groupby("split")["is_late"].agg(
    total_orders="count",
    late_orders="sum",
    late_rate="mean"
)

split_summary["late_rate"] *= 100

display(split_summary)

In [ ]:
feature_cols = [
    "order_value",
    "freight_value",
    "item_count",
    "n_sellers",
    "total_weight_g",
    "estimated_lead_days",
    "purchase_hour",
    "purchase_weekday",
    "purchase_month",
    "customer_state",
    "seller_state",
    "product_category",
    "same_state"
]

assert model_df["order_id"].is_unique
assert model_df["is_late"].isin([0, 1]).all()
assert model_df["split"].isin(
    ["train", "validation", "test"]
).all()

assert model_df[feature_cols].shape[1] == 13

# Save modeling dataset
output_path = PROCESSED_DIR / "late_delivery_modeling.parquet"

model_df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", model_df.shape)

In [ ]:
# Time when validation period begins
train_cutoff = model_df.loc[
    model_df["split"] == "validation",
    "order_purchase_timestamp"
].min()

# Get actual delivery time from original dataframe
delivery_times = model_df["order_id"].map(
    df.set_index("order_id")[
        "order_delivered_customer_date"
    ]
)

# Training orders with observed outcomes
train_mask = (
    (model_df["split"] == "train")
    & (delivery_times <= train_cutoff)
)

excluded_count = (
    (model_df["split"] == "train").sum()
    - train_mask.sum()
)

print("Training cutoff:", train_cutoff)
print("Eligible train orders:", train_mask.sum())
print("Excluded (label not yet available):", excluded_count)

In [ ]:
numeric_cols = [
    "order_value",
    "freight_value",
    "item_count",
    "n_sellers",
    "total_weight_g",
    "estimated_lead_days",
    "purchase_hour",
    "purchase_month",
    "same_state"
]

categorical_cols = [
    "customer_state",
    "seller_state",
    "product_category",
    "purchase_weekday"
]

features = numeric_cols + categorical_cols
target = "is_late"

# Split masks
val_mask = model_df["split"] == "validation"
test_mask = model_df["split"] == "test"

# Features
X_train = model_df.loc[train_mask, features].copy()
X_val = model_df.loc[val_mask, features].copy()
X_test = model_df.loc[test_mask, features].copy()

# Target
y_train = model_df.loc[train_mask, target].copy()
y_val = model_df.loc[val_mask, target].copy()
y_test = model_df.loc[test_mask, target].copy()

# Ensure numeric columns have numeric types
for X in [X_train, X_val, X_test]:
    X[numeric_cols] = X[numeric_cols].apply(
        pd.to_numeric, errors="raise"
    )

    # Ensure consistent categorical missing values
    X[categorical_cols] = (
        X[categorical_cols]
        .astype(object)
        .where(X[categorical_cols].notna(), np.nan)
    )

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

assert not set(X_train.index) & set(X_val.index)
assert not set(X_train.index) & set(X_test.index)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline ready!")

In [ ]:
# Learn preprocessing parameters ONLY from train
X_train_prepared = preprocessor.fit_transform(X_train)

# Apply learned transformations
X_val_prepared = preprocessor.transform(X_val)
X_test_prepared = preprocessor.transform(X_test)

print("Train:", X_train_prepared.shape)
print("Validation:", X_val_prepared.shape)
print("Test:", X_test_prepared.shape)

In [ ]:
from scipy import sparse

# Get feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

print("Original features:", len(features))
print("Transformed features:", len(feature_names))

# Check missing / invalid values
for name, X in [
    ("Train", X_train_prepared),
    ("Validation", X_val_prepared),
    ("Test", X_test_prepared)
]:
    values = X.data if sparse.issparse(X) else X

    print(
        name,
        "| Missing:", np.isnan(values).sum(),
        "| Infinite:", np.isinf(values).sum()
    )

# Display first 5 transformed rows
preview = X_train_prepared[:5]

if sparse.issparse(preview):
    preview = preview.toarray()

display(
    pd.DataFrame(
        preview,
        columns=feature_names
    ).T.head(25)
)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "Dummy": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),

    "Logistic Regression Balanced": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=100,
        class_weight="balanced",
        random_state=42
    )
}

In [ ]:
trained_models = {}

for name, model in models.items():

    print(f"Training {name}...")

    model.fit(X_train_prepared, y_train)

    trained_models[name] = model

    print("Done!\n")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_val_prepared)

    y_prob = model.predict_proba(
        X_val_prepared
    )[:, 1]

    results.append({
        "Model": name,

        "Accuracy": accuracy_score(
            y_val, y_pred
        ),

        "Precision": precision_score(
            y_val, y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_val, y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_val, y_pred,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_val, y_prob
        ),

        "PR-AUC": average_precision_score(
            y_val, y_prob
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df
    .sort_values("PR-AUC", ascending=False)
    .round(4)
)

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import (
    ConfusionMatrixDisplay
)

for name, model in trained_models.items():

    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay.from_estimator(
        model,
        X_val_prepared,
        y_val,
        display_labels=["On Time", "Late"],
        cmap="Blues",
        ax=ax
    )

    ax.set_title(name)

    plt.show()

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix
)

threshold_results = []
best_thresholds = {}

model_names = [
    "Logistic Regression",
    "Logistic Regression Balanced"
]

for name in model_names:

    model = trained_models[name]

    # Predicted probabilities
    y_prob = model.predict_proba(
        X_val_prepared
    )[:, 1]

    precision, recall, thresholds = (
        precision_recall_curve(y_val, y_prob)
    )

    # Calculate F1 for every threshold
    p = precision[:-1]
    r = recall[:-1]

    f1_scores = np.divide(
        2 * p * r,
        p + r,
        out=np.zeros_like(p),
        where=(p + r) > 0
    )

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]

    best_thresholds[name] = best_threshold

    # Predict with optimized threshold
    y_pred = (
        y_prob >= best_threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Model": name,
        "Threshold": best_threshold,
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        "F1", ascending=False
    ).round(4)
)

In [ ]:
plt.figure(figsize=(8, 6))

for name in model_names:

    model = trained_models[name]

    y_prob = model.predict_proba(
        X_val_prepared
    )[:, 1]

    precision, recall, _ = precision_recall_curve(
        y_val, y_prob
    )

    plt.plot(
        recall,
        precision,
        label=name
    )

# Baseline precision
plt.axhline(
    y=y_val.mean(),
    linestyle="--",
    color="gray",
    label="Baseline"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
selected_model = trained_models["Logistic Regression"]

selected_threshold = best_thresholds[
    "Logistic Regression"
]

y_val_prob = selected_model.predict_proba(
    X_val_prepared
)[:, 1]

y_val_pred = (
    y_val_prob >= selected_threshold
).astype(int)

print("Selected model: Logistic Regression")
print("Threshold:", round(selected_threshold, 4))
print("F1:", round(f1_score(y_val, y_val_pred), 4))

In [ ]:
# Extract feature coefficients
coefficients = pd.DataFrame({
    "feature": preprocessor.get_feature_names_out(),
    "coefficient": selected_model.coef_[0]
})

coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

# Top positive coefficients
top_positive = coefficients.nlargest(
    10, "coefficient"
)

# Top negative coefficients
top_negative = coefficients.nsmallest(
    10, "coefficient"
)

display(top_positive)
display(top_negative)

In [ ]:
important_features = pd.concat([
    top_positive,
    top_negative
]).sort_values("coefficient")

plt.figure(figsize=(10, 8))

colors = [
    "tomato" if x > 0 else "steelblue"
    for x in important_features["coefficient"]
]

plt.barh(
    important_features["feature"],
    important_features["coefficient"],
    color=colors
)

plt.axvline(0, color="black", linewidth=0.8)

plt.xlabel("Logistic Regression Coefficient")
plt.title("Features Associated with Late Delivery")

plt.tight_layout()
plt.show()

In [ ]:
# Create validation analysis dataframe
error_df = X_val.copy()

error_df["actual"] = y_val.to_numpy()
error_df["predicted"] = y_val_pred
error_df["late_probability"] = y_val_prob

# Classify prediction outcomes
conditions = [
    (error_df["actual"] == 1)
    & (error_df["predicted"] == 1),

    (error_df["actual"] == 0)
    & (error_df["predicted"] == 1),

    (error_df["actual"] == 1)
    & (error_df["predicted"] == 0)
]

error_df["result"] = np.select(
    conditions,
    ["TP", "FP", "FN"],
    default="TN"
)

display(error_df["result"].value_counts())

# Inspect false positives
display(
    error_df[
        error_df["result"] == "FP"
    ].head(10)
)

# Inspect false negatives
display(
    error_df[
        error_df["result"] == "FN"
    ].head(10)
)

In [ ]:
error_df["tp"] = (
    (error_df["actual"] == 1)
    & (error_df["predicted"] == 1)
).astype(int)

state_analysis = (
    error_df.groupby("customer_state")
    .agg(
        total_orders=("actual", "size"),
        late_orders=("actual", "sum"),
        predicted_late=("predicted", "sum"),
        correctly_detected=("tp", "sum")
    )
)

state_analysis["late_rate"] = (
    state_analysis["late_orders"]
    / state_analysis["total_orders"]
)

state_analysis["recall"] = (
    state_analysis["correctly_detected"]
    / state_analysis["late_orders"].replace(0, np.nan)
)

state_analysis["precision"] = (
    state_analysis["correctly_detected"]
    / state_analysis["predicted_late"].replace(0, np.nan)
)

# Focus on states with sufficient observations
display(
    state_analysis[
        state_analysis["total_orders"] >= 200
    ].sort_values(
        "late_rate", ascending=False
    ).round(4)
)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score

# Combine the already-fitted preprocessor and model
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", selected_model)
])

# Baseline PR-AUC on validation
baseline_ap = average_precision_score(
    y_val,
    full_pipeline.predict_proba(X_val)[:, 1]
)

print(f"Baseline PR-AUC: {baseline_ap:.4f}")

# Permute ORIGINAL features
perm_result = permutation_importance(
    full_pipeline,
    X_val,
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=1
)

# Summarize results
importance_df = pd.DataFrame({
    "Feature": X_val.columns,
    "Importance": perm_result.importances_mean,
    "Std": perm_result.importances_std
})

importance_df = importance_df.sort_values(
    "Importance", ascending=False
).reset_index(drop=True)

display(importance_df.round(5))

In [ ]:
plot_df = importance_df.sort_values("Importance")

plt.figure(figsize=(9, 6))

plt.barh(
    plot_df["Feature"],
    plot_df["Importance"],
    xerr=plot_df["Std"],
    capsize=3
)

plt.axvline(
    0,
    color="gray",
    linestyle="--"
)

plt.xlabel("Mean Decrease in PR-AUC")
plt.ylabel("Feature")
plt.title("Permutation Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Define feature groups
feature_groups = {
    "Geography": [
        "customer_state",
        "seller_state",
        "same_state"
    ],
    "Delivery Promise": [
        "estimated_lead_days"
    ],
    "Product Category": [
        "product_category"
    ],
    "Order & Package": [
        "order_value",
        "freight_value",
        "item_count",
        "n_sellers",
        "total_weight_g"
    ],
    "Purchase Time": [
        "purchase_hour",
        "purchase_month",
        "purchase_weekday"
    ]
}

# Define experiments
experiments = {
    "All Features": features.copy()
}

for group_name, group_features in feature_groups.items():

    experiments[f"Without {group_name}"] = [
        f for f in features
        if f not in group_features
    ]

ablation_results = []

for exp_name, selected_features in experiments.items():

    # Select numerical and categorical columns
    selected_num = [
        f for f in numeric_cols
        if f in selected_features
    ]

    selected_cat = [
        f for f in categorical_cols
        if f in selected_features
    ]

    # Build preprocessing for this experiment
    transformers = []

    if selected_num:
        transformers.append((
            "num",
            clone(numeric_pipeline),
            selected_num
        ))

    if selected_cat:
        transformers.append((
            "cat",
            clone(categorical_pipeline),
            selected_cat
        ))

    # Build a fresh model pipeline
    pipeline = Pipeline([
        ("preprocessor", ColumnTransformer(transformers)),
        ("classifier", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ])

    # Fit ONLY on training data
    pipeline.fit(
        X_train[selected_features],
        y_train
    )

    # Evaluate on validation
    y_prob = pipeline.predict_proba(
        X_val[selected_features]
    )[:, 1]

    ablation_results.append({
        "Experiment": exp_name,
        "N_Features": len(selected_features),
        "PR-AUC": average_precision_score(
            y_val, y_prob
        ),
        "ROC-AUC": roc_auc_score(
            y_val, y_prob
        )
    })

ablation_df = pd.DataFrame(ablation_results)

# Compare against the full-feature model
full_ap = ablation_df.loc[
    ablation_df["Experiment"] == "All Features",
    "PR-AUC"
].iloc[0]

ablation_df["Delta_PR_AUC"] = (
    ablation_df["PR-AUC"] - full_ap
)

display(
    ablation_df.sort_values(
        "PR-AUC", ascending=False
    ).round(5)
)

In [ ]:
plot_ablation = ablation_df[
    ablation_df["Experiment"] != "All Features"
].sort_values("Delta_PR_AUC")

plt.figure(figsize=(9, 5))

plt.barh(
    plot_ablation["Experiment"],
    plot_ablation["Delta_PR_AUC"]
)

plt.axvline(
    0,
    color="gray",
    linestyle="--"
)

plt.xlabel("Change in PR-AUC vs Full Model")
plt.title("Feature Group Ablation Results")

plt.tight_layout()
plt.show()

In [ ]:
# Core features identified from Phase 4B
core_features = [
    "customer_state",
    "seller_state",
    "same_state",
    "estimated_lead_days"
]

X_train_fe = X_train.copy()
X_val_fe = X_val.copy()

# --------------------------------------------------
# Feature 1: Route Pair
# --------------------------------------------------

train_routes = (
    X_train_fe["seller_state"].astype(str)
    + "_"
    + X_train_fe["customer_state"].astype(str)
)

val_routes = (
    X_val_fe["seller_state"].astype(str)
    + "_"
    + X_val_fe["customer_state"].astype(str)
)

# Keep routes appearing at least 50 times in train
route_counts = train_routes.value_counts()
common_routes = route_counts[
    route_counts >= 50
].index

X_train_fe["route_pair"] = train_routes.where(
    train_routes.isin(common_routes), "OTHER"
)

X_val_fe["route_pair"] = val_routes.where(
    val_routes.isin(common_routes), "OTHER"
)

# --------------------------------------------------
# Feature 2: Estimated Lead Time Groups
# --------------------------------------------------

lead_bins = [-np.inf, 7, 14, 21, 30, 45, np.inf]
lead_labels = [
    "<=7", "7-14", "14-21",
    "21-30", "30-45", ">45"
]

for X in [X_train_fe, X_val_fe]:
    X["estimated_lead_group"] = pd.cut(
        X["estimated_lead_days"],
        bins=lead_bins,
        labels=lead_labels
    ).astype(object)

print("Common routes:", len(common_routes))

display(
    X_train_fe[
        ["route_pair", "estimated_lead_days",
         "estimated_lead_group"]
    ].head()
)

In [ ]:
from sklearn.base import clone

experiments_fe = {
    "E0 - Full 13": features.copy(),

    "E1 - Core 4": core_features,

    "E2 - Core + Route": (
        core_features + ["route_pair"]
    ),

    "E3 - Core + Lead Groups": (
        core_features + ["estimated_lead_group"]
    ),

    "E4 - Core + Both": (
        core_features
        + ["route_pair", "estimated_lead_group"]
    )
}

fe_results = []
fe_models = {}

for exp_name, cols in experiments_fe.items():

    selected_num = [
        c for c in cols if c in numeric_cols
    ]

    selected_cat = [
        c for c in cols if
        c in categorical_cols
        or c in ["route_pair", "estimated_lead_group"]
    ]

    transformers = [
        ("num", clone(numeric_pipeline), selected_num),
        ("cat", clone(categorical_pipeline), selected_cat)
    ]

    pipeline = Pipeline([
        ("preprocessor", ColumnTransformer(transformers)),
        ("classifier", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ])

    pipeline.fit(
        X_train_fe[cols],
        y_train
    )

    y_prob = pipeline.predict_proba(
        X_val_fe[cols]
    )[:, 1]

    fe_models[exp_name] = pipeline

    fe_results.append({
        "Experiment": exp_name,
        "N_Features": len(cols),
        "PR-AUC": average_precision_score(
            y_val, y_prob
        ),
        "ROC-AUC": roc_auc_score(
            y_val, y_prob
        )
    })

fe_results_df = pd.DataFrame(fe_results)

baseline_ap = fe_results_df.loc[
    fe_results_df["Experiment"] == "E0 - Full 13",
    "PR-AUC"
].iloc[0]

fe_results_df["Delta_PR_AUC"] = (
    fe_results_df["PR-AUC"] - baseline_ap
)

display(
    fe_results_df.sort_values(
        "PR-AUC", ascending=False
    ).round(5)
)

In [ ]:
plot_df = fe_results_df.sort_values("PR-AUC")

plt.figure(figsize=(9, 5))

plt.barh(
    plot_df["Experiment"],
    plot_df["PR-AUC"]
)

plt.axvline(
    baseline_ap,
    linestyle="--",
    color="red",
    label=f"Baseline: {baseline_ap:.4f}"
)

plt.xlabel("Validation PR-AUC")
plt.title("Feature Engineering Experiment Comparison")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score
)

candidate_names = [
    "E0 - Full 13",
    "E1 - Core 4",
    "E3 - Core + Lead Groups"
]

comparison_results = []
optimal_thresholds = {}

for name in candidate_names:

    model = fe_models[name]
    cols = experiments_fe[name]

    y_prob = model.predict_proba(
        X_val_fe[cols]
    )[:, 1]

    precision, recall, thresholds = (
        precision_recall_curve(y_val, y_prob)
    )

    p = precision[:-1]
    r = recall[:-1]

    f1_values = np.divide(
        2 * p * r,
        p + r,
        out=np.zeros_like(p),
        where=(p + r) > 0
    )

    best_idx = np.argmax(f1_values)
    best_threshold = thresholds[best_idx]

    optimal_thresholds[name] = best_threshold

    y_pred = (
        y_prob >= best_threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, y_pred, labels=[0, 1]
    ).ravel()

    comparison_results.append({
        "Model": name,
        "Threshold": best_threshold,
        "PR-AUC": average_precision_score(
            y_val, y_prob
        ),
        "Precision": precision_score(
            y_val, y_pred
        ),
        "Recall": recall_score(
            y_val, y_pred
        ),
        "F1": f1_score(y_val, y_pred),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Alert_Rate": y_pred.mean()
    })

comparison_df = pd.DataFrame(
    comparison_results
)

display(
    comparison_df.sort_values(
        "F1", ascending=False
    ).round(4)
)

In [ ]:
monthly_df = pd.DataFrame({
    "purchase_date": model_df.loc[
        X_val_fe.index,
        "order_purchase_timestamp"
    ],
    "actual": y_val
}, index=X_val_fe.index)

monthly_df["month"] = (
    pd.to_datetime(monthly_df["purchase_date"])
    .dt.to_period("M")
    .astype(str)
)

for name in candidate_names:

    cols = experiments_fe[name]

    monthly_df[name] = fe_models[name].predict_proba(
        X_val_fe[cols]
    )[:, 1]

monthly_results = []

for month, group in monthly_df.groupby("month"):

    row = {
        "Month": month,
        "Orders": len(group),
        "Late_Orders": group["actual"].sum(),
        "Late_Rate": group["actual"].mean()
    }

    for name in candidate_names:

        if group["actual"].nunique() == 2:
            ap = average_precision_score(
                group["actual"],
                group[name]
            )
        else:
            ap = np.nan

        row[name] = ap

    monthly_results.append(row)

monthly_results_df = pd.DataFrame(monthly_results)

display(monthly_results_df.round(4))

In [ ]:
# Prepare test data
X_test_fe = X_test.copy()

X_test_fe["estimated_lead_group"] = pd.cut(
    X_test_fe["estimated_lead_days"],
    bins=lead_bins,
    labels=lead_labels
).astype(object)

# Models selected before test evaluation
final_candidates = [
    "E0 - Full 13",
    "E3 - Core + Lead Groups"
]

for name in final_candidates:
    cols = experiments_fe[name]

    assert X_test_fe[cols].shape[1] == len(cols)
    print(name, ":", X_test_fe[cols].shape)

print("Test observations:", len(y_test))
print("Actual late rate:", round(y_test.mean(), 4))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

test_results = []
test_predictions = {}

for name in final_candidates:

    model = fe_models[name]
    cols = experiments_fe[name]

    threshold = optimal_thresholds[name]

    # Predict on untouched test data
    y_prob = model.predict_proba(
        X_test_fe[cols]
    )[:, 1]

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    test_predictions[name] = {
        "prob": y_prob,
        "pred": y_pred
    }

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    test_results.append({
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(
            y_test, y_prob
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Alert_Rate": y_pred.mean()
    })

test_results_df = pd.DataFrame(test_results)

display(test_results_df.round(4))

In [ ]:
comparison_rows = []

for name in final_candidates:

    val_row = comparison_df[
        comparison_df["Model"] == name
    ].iloc[0]

    test_row = test_results_df[
        test_results_df["Model"] == name
    ].iloc[0]

    comparison_rows.append({
        "Model": name,
        "Val_PR_AUC": val_row["PR-AUC"],
        "Test_PR_AUC": test_row["PR-AUC"],
        "Val_F1": val_row["F1"],
        "Test_F1": test_row["F1"],
        "Val_Recall": val_row["Recall"],
        "Test_Recall": test_row["Recall"],
        "Val_Precision": val_row["Precision"],
        "Test_Precision": test_row["Precision"]
    })

final_comparison = pd.DataFrame(comparison_rows)

display(final_comparison.round(4))

In [ ]:
from sklearn.metrics import precision_recall_curve

model = fe_models["E0 - Full 13"]
cols = experiments_fe["E0 - Full 13"]

# Validation probabilities
y_prob = model.predict_proba(
    X_val_fe[cols]
)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val, y_prob
)

# Last precision/recall point has no threshold
p = precision[:-1]
r = recall[:-1]

f1 = np.divide(
    2 * p * r,
    p + r,
    out=np.zeros_like(p),
    where=(p + r) > 0
)

# Fraction of orders flagged as late
alert_rate = np.array([
    np.mean(y_prob >= t)
    for t in thresholds
])

best_idx = np.argmax(f1)

print("Best threshold:", thresholds[best_idx])
print("Best F1:", f1[best_idx])
print("Precision:", p[best_idx])
print("Recall:", r[best_idx])
print("Alert rate:", alert_rate[best_idx])

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(thresholds, p, label="Precision")
plt.plot(thresholds, r, label="Recall")
plt.plot(thresholds, f1, label="F1-score")
plt.plot(thresholds, alert_rate, label="Alert Rate")

plt.axvline(
    thresholds[best_idx],
    color="black",
    linestyle="--",
    label="Validation F1-optimal"
)

plt.xlim(0, 0.5)
plt.xlabel("Decision Threshold")
plt.ylabel("Metric Value")
plt.title("E0 - Threshold Sensitivity")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Final selected model for reporting
final_name = "E0 - Full 13"

final_model = fe_models[final_name]
final_features = experiments_fe[final_name]
final_threshold = optimal_thresholds[final_name]

# Predictions already obtained in Phase 4C
final_prob = test_predictions[final_name]["prob"]
final_pred = test_predictions[final_name]["pred"]

# Final metrics
final_metrics = test_results_df[
    test_results_df["Model"] == final_name
].copy()

# Baseline prevalence
test_late_rate = y_test.mean()

final_metrics["Baseline_AP"] = test_late_rate

final_metrics["AP_Lift"] = (
    final_metrics["PR-AUC"] / test_late_rate
)

print("Selected model:", final_name)
print("Threshold:", round(final_threshold, 4))
print("Test late rate:", round(test_late_rate, 4))

display(final_metrics.T)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_pred,
    labels=[0, 1],
    display_labels=["On Time", "Late"],
    cmap="Blues",
    values_format="d",
    ax=ax
)

ax.set_title("E0 - Final Test Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

precision, recall, _ = precision_recall_curve(
    y_test,
    final_prob
)

ap = average_precision_score(
    y_test,
    final_prob
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision,
    label=f"E0 (AP = {ap:.4f})"
)

plt.axhline(
    y_test.mean(),
    color="gray",
    linestyle="--",
    label=f"Random Baseline = {y_test.mean():.4f}"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("E0 - Precision-Recall Curve on Test Set")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
generalization_df = final_comparison.copy()

generalization_df["AP_Drop"] = (
    generalization_df["Val_PR_AUC"]
    - generalization_df["Test_PR_AUC"]
)

generalization_df["AP_Relative_Drop_%"] = (
    generalization_df["AP_Drop"]
    / generalization_df["Val_PR_AUC"] * 100
)

generalization_df["F1_Drop"] = (
    generalization_df["Val_F1"]
    - generalization_df["Test_F1"]
)

display(generalization_df.round(4))

In [ ]:
# Build test analysis dataframe
test_analysis = model_df.loc[
    X_test.index,
    [
        "order_id",
        "order_purchase_timestamp",
        "customer_state"
    ]
].copy()

assert X_test.index.equals(y_test.index)

test_analysis["actual"] = y_test.to_numpy()
test_analysis["predicted"] = final_pred
test_analysis["score"] = final_prob

test_analysis["TP"] = (
    (test_analysis["actual"] == 1)
    & (test_analysis["predicted"] == 1)
).astype(int)

test_analysis["FP"] = (
    (test_analysis["actual"] == 0)
    & (test_analysis["predicted"] == 1)
).astype(int)

# Summary by customer state
state_results = (
    test_analysis.groupby("customer_state")
    .agg(
        Orders=("actual", "size"),
        Late=("actual", "sum"),
        TP=("TP", "sum"),
        FP=("FP", "sum")
    )
)

state_results["Late_Rate"] = (
    state_results["Late"]
    / state_results["Orders"]
)

state_results["Recall"] = (
    state_results["TP"]
    / state_results["Late"].replace(0, np.nan)
)

state_results["Precision"] = (
    state_results["TP"]
    / (state_results["TP"] + state_results["FP"])
      .replace(0, np.nan)
)

# Only show groups with enough observations
display(
    state_results[
        (state_results["Orders"] >= 200)
        & (state_results["Late"] >= 10)
    ].sort_values(
        "Recall", ascending=False
    ).round(4)
)